# EDA Fraud Splits

Train/validation/test split analysis for fraud datasets.

Steps:
- Load creditcard data and compute split sizes.
- Verify class balance per split.
- Report split statistics for reference.
- Run the training data audit and summarize outputs.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
import pandas as pd
from pathlib import Path
from uais.data.split_data import split_train_val_test

fraud_root = REPO_ROOT / 'data' / 'raw' / 'fraud'
creditcard_path = fraud_root / 'creditcard.csv'

summary = {
    'creditcard': {},
}

if not creditcard_path.exists():
    print('Missing creditcard.csv:', creditcard_path)
else:
    df = pd.read_csv(creditcard_path)
    summary['creditcard']['rows'] = int(df.shape[0])
    summary['creditcard']['cols'] = int(df.shape[1])
    print('Dataset shape:', df.shape)

    if 'Class' not in df.columns:
        print('Missing Class column for labels')
    else:
        X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
            df, target='Class', test_size=0.2, val_size=0.1, random_state=42
        )
        summary['creditcard']['split_sizes'] = {
            'train': int(len(y_train)),
            'val': int(len(y_val)),
            'test': int(len(y_test)),
        }
        def ratio(series):
            counts = series.value_counts().to_dict()
            total = sum(counts.values())
            fraud_ratio = counts.get(1, 0) / total if total else 0
            return {'counts': counts, 'fraud_ratio': round(fraud_ratio, 6)}

        summary['creditcard']['train'] = ratio(y_train)
        summary['creditcard']['val'] = ratio(y_val)
        summary['creditcard']['test'] = ratio(y_test)

        print('Split sizes:', summary['creditcard']['split_sizes'])
        print('Train distribution:', summary['creditcard']['train'])
        print('Val distribution:', summary['creditcard']['val'])
        print('Test distribution:', summary['creditcard']['test'])


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'eda_fraud_splits_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Generate a training data audit
run([PY, 'scripts/training_data_audit.py'])



In [ ]:
# Summarize fraud-related entries from the training data audit.
audit_path = REPO_ROOT / 'reports' / 'TRAINING_DATA.json'
if not audit_path.exists():
    print('Missing:', audit_path)
else:
    audit = json.loads(audit_path.read_text(encoding='utf-8'))
    items = [
        item for item in audit.get('required', []) + audit.get('optional', [])
        if 'fraud' in str(item.get('name', '')).lower()
    ]
    if not items:
        print('No fraud entries found in TRAINING_DATA.json')
    else:
        print('fraud datasets in audit:')
        for item in items:
            print(' -', item.get('name'), '|', item.get('status'), '|', item.get('path'))


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
